#  Nettoyage des Données

## Objectif
Ce notebook nettoie et prépare les données brutes pour la modélisation.  

## Étapes
1. Chargement des données brutes
2. Filtrage des commandes livrées
3. Fusion de toutes les tables
4. Suppression des valeurs manquantes
5. Traduction des catégories en français
6. Vérification finale et export

## 1. Chargement des données brutes

On recharge les 4 tables depuis les fichiers CSV originaux.  

In [1]:
# ============================================================
# IMPORTS ET CHARGEMENT DES DONNÉES
# ============================================================

import pandas as pd
import numpy as np

# Chargement des 4 tables
orders   = pd.read_csv('../data/olist_orders_dataset.csv')
items    = pd.read_csv('../data/olist_order_items_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')
reviews  = pd.read_csv('../data/olist_order_reviews_dataset.csv')

print(" Données chargées")
print(f"   - orders   : {orders.shape[0]:,} lignes")
print(f"   - items    : {items.shape[0]:,} lignes")
print(f"   - products : {products.shape[0]:,} lignes")
print(f"   - reviews  : {reviews.shape[0]:,} lignes")

 Données chargées
   - orders   : 99,441 lignes
   - items    : 112,650 lignes
   - products : 32,951 lignes
   - reviews  : 99,224 lignes


## 2. Filtrage des commandes livrées

On garde uniquement les commandes avec le statut "delivered".  
Les commandes annulées, en cours ou non disponibles ne représentent  
pas de vraies transactions finalisées et pourraient fausser notre analyse.

In [2]:
# ============================================================
# FILTRAGE DES COMMANDES LIVRÉES
# ============================================================

# Avant filtrage
print(f"Commandes avant filtrage : {orders.shape[0]:,}")
print(f"Statuts présents : {orders['order_status'].unique()}")

# On garde uniquement les commandes livrées
orders_clean = orders[orders['order_status'] == 'delivered'].copy()

# Après filtrage
print(f"\nCommandes après filtrage : {orders_clean.shape[0]:,}")
print(f"Commandes supprimées     : {orders.shape[0] - orders_clean.shape[0]:,}")
print(f"Statuts restants         : {orders_clean['order_status'].unique()}")

Commandes avant filtrage : 99,441
Statuts présents : <StringArray>
[  'delivered',    'invoiced',     'shipped',  'processing', 'unavailable',
    'canceled',     'created',    'approved']
Length: 8, dtype: str

Commandes après filtrage : 96,478
Commandes supprimées     : 2,963
Statuts restants         : <StringArray>
['delivered']
Length: 1, dtype: str


## 3. Fusion de toutes les tables

On relie toutes les tables ensemble via leurs colonnes communes.  
L'objectif est d'avoir dans un seul DataFrame :
- L'identifiant de commande et son statut → `orders`
- Le prix de chaque produit → `items`
- La catégorie de chaque produit → `products`
- La note du client → `reviews`

Schéma des relations :
orders ──(order_id)──> items ──(product_id)──> products
orders ──(order_id)──> reviews

In [3]:
# ============================================================
# FUSION DE TOUTES LES TABLES
# ============================================================

# Étape 1 : orders + items (via order_id)
df = orders_clean[['order_id']].merge(
    items[['order_id', 'product_id', 'price', 'freight_value']],
    on='order_id',
    how='inner'  # On garde uniquement les lignes présentes dans les deux tables
)
print(f"Étape 1 - orders + items    : {df.shape[0]:,} lignes")

# Étape 2 : + products (via product_id)
df = df.merge(
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
)
print(f"Étape 2 - + products        : {df.shape[0]:,} lignes")

# Étape 3 : + reviews (via order_id)
df = df.merge(
    reviews[['order_id', 'review_score']],
    on='order_id',
    how='left'
)
print(f"Étape 3 - + reviews         : {df.shape[0]:,} lignes")

print(f"\n Fusion terminée !")
print(f"   Colonnes disponibles : {df.columns.tolist()}")

Étape 1 - orders + items    : 110,197 lignes
Étape 2 - + products        : 110,197 lignes
Étape 3 - + reviews         : 110,840 lignes

 Fusion terminée !
   Colonnes disponibles : ['order_id', 'product_id', 'price', 'freight_value', 'product_category_name', 'review_score']


On remarque a l'étape 3 après la jointure de la table reviews, le nombre de lignes a augmenté. Ceci s'explique probablement par le fait que certaines commandes ont plusieurs avis dans la table reviews. 

On va revoir notre code pour garder uniquement le dernier avis par commande, ceci est important pour la modélisation.

In [4]:
# ============================================================
# FUSION DE TOUTES LES TABLES
# ============================================================

# Étape 1 : orders + items (via order_id)
df = orders_clean[['order_id']].merge(
    items[['order_id', 'product_id', 'price', 'freight_value']],
    on='order_id',
    how='inner'
)
print(f"Étape 1 - orders + items    : {df.shape[0]:,} lignes")

# Étape 2 : + products (via product_id)
df = df.merge(
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
)
print(f"Étape 2 - + products        : {df.shape[0]:,} lignes")

# Étape 3 : + reviews (via order_id)
# On garde uniquement le dernier avis par commande pour éviter les doublons
reviews_dedup = reviews.sort_values('review_creation_date').drop_duplicates(
    subset='order_id', 
    keep='last'  # Garde le dernier avis si plusieurs existent
)

df = df.merge(
    reviews_dedup[['order_id', 'review_score']],
    on='order_id',
    how='left'
)
print(f"Étape 3 - + reviews         : {df.shape[0]:,} lignes")

print(f"\n Fusion terminée !")
print(f"   Colonnes disponibles : {df.columns.tolist()}")

Étape 1 - orders + items    : 110,197 lignes
Étape 2 - + products        : 110,197 lignes
Étape 3 - + reviews         : 110,197 lignes

 Fusion terminée !
   Colonnes disponibles : ['order_id', 'product_id', 'price', 'freight_value', 'product_category_name', 'review_score']


## 4. Suppression des valeurs manquantes

On identifie et supprime les lignes incomplètes qui pourraient  
fausser notre analyse :
- Produits sans catégorie donc impossible de faire une analyse par catégorie.

In [5]:
# ============================================================
# SUPPRESSION DES VALEURS MANQUANTES
# ============================================================

print("=== Valeurs manquantes avant nettoyage ===")
print(df.isnull().sum())

# On supprime uniquement les lignes sans catégorie
# car c'est indispensable pour notre analyse
df_clean = df.dropna(subset=['product_category_name']).copy()

print(f"\n=== Résultat ===")
print(f"Lignes avant : {df.shape[0]:,}")
print(f"Lignes après : {df_clean.shape[0]:,}")
print(f"Lignes supprimées : {df.shape[0] - df_clean.shape[0]:,}")

print(f"\n=== Valeurs manquantes après nettoyage ===")
print(df_clean.isnull().sum())

=== Valeurs manquantes avant nettoyage ===
order_id                    0
product_id                  0
price                       0
freight_value               0
product_category_name    1537
review_score              827
dtype: int64

=== Résultat ===
Lignes avant : 110,197
Lignes après : 108,660
Lignes supprimées : 1,537

=== Valeurs manquantes après nettoyage ===
order_id                   0
product_id                 0
price                      0
freight_value              0
product_category_name      0
review_score             815
dtype: int64


> **Observations :**
> - 1 537 lignes supprimées car sans catégorie (indispensable pour notre analyse)
> - 815 avis manquants conservés : la note client est utile mais pas critique
> - Base de travail finale : 108 660 lignes

## 5. Traduction des catégories en français

Les catégories sont actuellement en portugais, ce qui rend  
les graphiques difficiles à lire. On les traduit en français  
pour rendre le projet plus accessible.

In [6]:
# ============================================================
# TRADUCTION DES CATÉGORIES EN FRANÇAIS
# ============================================================

# Dictionnaire de traduction portugais -> français
traduction = {
    'cama_mesa_banho'                            : 'Literie & Salle de bain',
    'esporte_lazer'                              : 'Sport & Loisirs',
    'moveis_decoracao'                           : 'Meubles & Décoration',
    'beleza_saude'                               : 'Beauté & Santé',
    'utilidades_domesticas'                      : 'Ustensiles Ménagers',
    'automotivo'                                 : 'Automobile',
    'informatica_acessorios'                     : 'Informatique',
    'brinquedos'                                 : 'Jouets',
    'relogios_presentes'                         : 'Montres & Cadeaux',
    'telefonia'                                  : 'Téléphonie',
    'bebes'                                      : 'Bébé',
    'perfumaria'                                 : 'Parfumerie',
    'papelaria'                                  : 'Papeterie',
    'fashion_bolsas_e_acessorios'                : 'Mode & Accessoires',
    'cool_stuff'                                 : 'Gadgets',
    'eletronicos'                                : 'Électronique',
    'ferramentas_jardim'                         : 'Outils & Jardin',
    'pcs'                                        : 'Ordinateurs',
    'climatizacao'                               : 'Climatisation',
    'livros_tecnicos'                            : 'Livres Techniques',
    'musica'                                     : 'Musique',
    'construcao_ferramentas_seguranca'           : 'Construction & Sécurité',
    'eletrodomesticos'                           : 'Électroménager',
    'eletrodomesticos_2'                         : 'Électroménager 2',
    'eletroportateis'                            : 'Appareils Portables',
    'instrumentos_musicais'                      : 'Instruments de Musique',
    'telefonia_fixa'                             : 'Téléphonie Fixe',
    'fashion_roupa_masculina'                    : 'Mode Homme',
    'fashion_roupa_feminina'                     : 'Mode Femme',
    'alimentos_bebidas'                          : 'Alimentation & Boissons',
    'alimentos'                                  : 'Alimentation',
    'livros_interesse_geral'                     : 'Livres Général',
    'livros_importados'                          : 'Livres Importés',
    'dvds_blu_ray'                               : 'DVD & Blu-ray',
    'cds_dvds_musicais'                          : 'CD & DVD Musicaux',
    'consoles_games'                             : 'Consoles & Jeux',
    'pc_gamer'                                   : 'PC Gamer',
    'tablets_impressao_imagem'                   : 'Tablettes & Impression',
    'moveis_escritorio'                          : 'Mobilier Bureau',
    'moveis_sala'                                : 'Mobilier Salon',
    'moveis_quarto'                              : 'Mobilier Chambre',
    'moveis_colchao_e_estofado'                  : 'Matelas & Canapés',
    'moveis_cozinha_area_de_servico_jantar_e_jardim' : 'Mobilier Cuisine & Jardin',
    'portateis_casa_forno_e_cafe'                : 'Fours & Cafetières',
    'portateis_cozinha_e_preparadores_de_alimentos'  : 'Cuisine & Préparation',
    'agro_industria_e_comercio'                  : 'Agriculture & Commerce',
    'industria_comercio_e_negocios'              : 'Industrie & Commerce',
    'sinalizacao_e_seguranca'                    : 'Signalisation & Sécurité',
    'construcao_ferramentas_construcao'          : 'Outils Construction',
    'construcao_ferramentas_iluminacao'          : 'Éclairage Construction',
    'construcao_ferramentas_jardim'              : 'Outils Jardin',
    'artigos_de_natal'                           : 'Articles de Noël',
    'artigos_de_festas'                          : 'Articles de Fête',
    'fashion_esporte'                            : 'Mode Sport',
    'fashion_underwear_e_moda_praia'             : 'Lingerie & Maillots',
    'fashion_roupa_infanto_juvenil'              : 'Mode Enfant',
    'fraldas_higiene'                            : 'Couches & Hygiène',
    'la_cuisine'                                 : 'Cuisine',
    'pet_shop'                                   : 'Animalerie',
    'flores'                                     : 'Fleurs',
    'artes'                                      : 'Arts',
    'artes_e_artesanato'                         : 'Arts & Artisanat',
    'casa_conforto'                              : 'Confort Maison',
    'casa_conforto_2'                            : 'Confort Maison 2',
    'casa_construcao'                            : 'Construction Maison',
    'seguros_e_servicos'                         : 'Assurances & Services',
    'audio'                                      : 'Audio',
    'market_place'                               : 'Marketplace',
    'recarga_saldo'                              : 'Recharge',
    'portateis_cozinha_e_preparadores_de_alimentos' : 'Cuisine Portable',
}

# Appliquer la traduction
df_clean['categorie'] = df_clean['product_category_name'].map(traduction)

# Vérifier les catégories non traduites
non_traduits = df_clean[df_clean['categorie'].isna()]['product_category_name'].unique()
print(f"  Catégories non traduites : {len(non_traduits)}")
if len(non_traduits) > 0:
    print(non_traduits)

# Statistiques
print(f"\n Traduction terminée")
print(f"   Catégories traduites : {df_clean['categorie'].nunique()}")

  Catégories non traduites : 5
<StringArray>
[                  'malas_acessorios',                            'bebidas',
                   'fashion_calcados',                          'cine_foto',
 'construcao_ferramentas_ferramentas']
Length: 5, dtype: str

 Traduction terminée
   Catégories traduites : 68


### Correction des catégories manquantes

5 catégories n'ont pas été traduites, on les ajoute manuellement.

In [7]:
# ============================================================
# AJOUT DES CATÉGORIES MANQUANTES
# ============================================================

traduction_manquante = {
    'malas_acessorios'                       : 'Valises & Accessoires',
    'bebidas'                                : 'Boissons',
    'fashion_calcados'                       : 'Chaussures',
    'cine_foto'                              : 'Cinéma & Photo',
    'construcao_ferramentas_ferramentas'     : 'Outils & Bricolage',
}

# Appliquer les traductions manquantes
df_clean['categorie'] = df_clean['categorie'].fillna(
    df_clean['product_category_name'].map(traduction_manquante)
)

# Vérification finale
non_traduits = df_clean[df_clean['categorie'].isna()]['product_category_name'].unique()
print(f"  Catégories encore non traduites : {len(non_traduits)}")
print(f" Nombre total de catégories      : {df_clean['categorie'].nunique()}")

  Catégories encore non traduites : 0
 Nombre total de catégories      : 73


## 6. Vérification finale et export

On vérifie une dernière fois la qualité du DataFrame final  
avant de l'exporter en CSV. C'est cette base qu'on va utilisé dans la suite de notre projet.

In [8]:
# ============================================================
# VÉRIFICATION FINALE
# ============================================================

print("=" * 55)
print("         BILAN DU NETTOYAGE")
print("=" * 55)

print(f"""
 DONNÉES
   - Lignes initiales  : 110,197
   - Lignes finales    : {df_clean.shape[0]:,}
   - Lignes supprimées : {110197 - df_clean.shape[0]:,}
   - Colonnes          : {df_clean.columns.tolist()}

 VÉRIFICATIONS
   - Valeurs manquantes (catégorie) : {df_clean['categorie'].isna().sum()}
   - Valeurs manquantes (prix)      : {df_clean['price'].isna().sum()}
   - Prix négatifs ou nuls          : {(df_clean['price'] <= 0).sum()}
   - Catégories disponibles         : {df_clean['categorie'].nunique()}

 PRIX
   - Prix moyen  : {df_clean['price'].mean():.2f} R$
   - Prix médian : {df_clean['price'].median():.2f} R$
   - Prix min    : {df_clean['price'].min():.2f} R$
   - Prix max    : {df_clean['price'].max():.2f} R$
""")

print("=" * 55)

         BILAN DU NETTOYAGE

 DONNÉES
   - Lignes initiales  : 110,197
   - Lignes finales    : 108,660
   - Lignes supprimées : 1,537
   - Colonnes          : ['order_id', 'product_id', 'price', 'freight_value', 'product_category_name', 'review_score', 'categorie']

 VÉRIFICATIONS
   - Valeurs manquantes (catégorie) : 0
   - Valeurs manquantes (prix)      : 0
   - Prix négatifs ou nuls          : 0
   - Catégories disponibles         : 73

 PRIX
   - Prix moyen  : 120.11 R$
   - Prix médian : 74.90 R$
   - Prix min    : 0.85 R$
   - Prix max    : 6735.00 R$



## 7. Export du fichier nettoyé

On exporte le DataFrame final en CSV.  
On supprime la colonne `product_category_name` (portugais) car on a maintenant la colonne `categorie` (français).

In [9]:
# ============================================================
# EXPORT DU FICHIER NETTOYÉ
# ============================================================

# On supprime la colonne portugaise devenue inutile
df_final = df_clean.drop(columns=['product_category_name'])

# Renommer les colonnes en français pour plus de clarté
df_final = df_final.rename(columns={
    'order_id'      : 'commande_id',
    'product_id'    : 'produit_id',
    'price'         : 'prix',
    'freight_value' : 'frais_livraison',
    'review_score'  : 'note_client',
    'categorie'     : 'categorie',
})

# Export en CSV
df_final.to_csv('../data/df_clean.csv', index=False)

print(" Fichier exporté : data/df_clean.csv")
print(f"   Shape final     : {df_final.shape[0]:,} lignes, {df_final.shape[1]} colonnes")
print(f"   Colonnes        : {df_final.columns.tolist()}")
print("\n Ce fichier sera utilisé dans tous les notebooks suivants !")

 Fichier exporté : data/df_clean.csv
   Shape final     : 108,660 lignes, 6 colonnes
   Colonnes        : ['commande_id', 'produit_id', 'prix', 'frais_livraison', 'note_client', 'categorie']

 Ce fichier sera utilisé dans tous les notebooks suivants !
